In [14]:
import json
import struct
import numpy as np
import matplotlib.pyplot as plt
import photonforge as pf
import siepic_forge as siepic
import luxtelligence_lnoi400_forge as lxt
import tidy3d as td

td.config.logging.level = "ERROR"

# Set up technologies
siepic_tech = siepic.ebeam()
basic_tech = pf.basic_technology()
lxt_tech = lxt.lnoi400()
pf.config.default_technology = siepic_tech

# Initialize live viewer for real-time visualization
from photonforge.live_viewer import LiveViewer
viewer = LiveViewer()

# Define simulation parameters
wavelengths = np.linspace(1.53, 1.57, 101)
freqs = pf.C_0 / wavelengths

22:51:46 SE Asia Standard Time WARNING: Using canonical configuration directory 
                               at 'C:\Users\James\.config\tidy3d'. Found legacy 
                               directory at 'C:\Users\James\.tidy3d', which will
                               be ignored. Tidy3D configuration now uses        
                               'C:\Users\James\.config\tidy3d\config.toml'.     

22:52:03 SE Asia Standard Time WARNING: The material-library variant            
                               'Palik_Lossless' is deprecated and maps to       
                               'Palik_LowLoss' because it contains a tiny fitted
                               loss despite its name. Use 'Palik_NoLoss' where  
                               available for a zero-loss Palik model.           

LiveViewer started at http://localhost:53211


In [15]:
dual_mode_spec = siepic_tech.ports["TE_1550_500"].copy()
dual_mode_spec.num_modes = 2  # Use both modes

siepic_tech.add_port("TE-TM_1550_500", dual_mode_spec)
siepic_tech.ports["TE-TM_1550_500"]

PortSpec(description="Strip TE 1550 nm, w=500 nm", width=1.5, limits=(-0.6, 0.82), num_modes=2, added_solver_modes=0, polarization="", target_neff=3.5, default_radius=0, path_profiles=[(0.5, 0, (1, 0))])

In [20]:
# digital metamaterial based on 0 and 1, 0 is air, 1 is silicon

# later on if we want to do inverse design we can sweep the num)
num = 10

bits = bin(num)[2:]
print(bits)
print(bits[1])

rot_length = 5.0
rot_width = 1.5
wg_resolution = 0.1
max_pixels = int(rot_length/wg_resolution) * int(rot_width/wg_resolution)
bits = bin(num)[2:].zfill(max_pixels)
print(bits)

1010
0
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001010


In [84]:
# digital metamaterial based on 0 and 1, 0 is air, 1 is silicon
# max pixels = rot_length/wg_resolution * rot_width/wg_resolution = 5.0/0.1 * 1.5/0.1 = 750

@pf.parametric_component
def create_hybrid_rotator_90(*, num = 10, rot_length = 5.0, rot_width = 1.5, wg_resolution = 0.1):

    wg_height = 0.22
    max_pixels = int(rot_length/wg_resolution) * int(rot_width/wg_resolution)

    # a reminder to connect to the waveguides on the left and right sides of the component
    #wg_length = 2.0
    #wg_width = 0.5

    # add the 0000 until the character count = max_pixels 
    bits = bin(num)[2:].zfill(max_pixels)

    # Create an empty component named "HybridRotator"
    hybrid_rotator_90 = pf.Component("HybridRotator90")

    # Create the digital metamaterial based on the binary representation of the number
    n = 0
    for i in range(int(rot_width/wg_resolution)):
        for j in range(int(rot_length/wg_resolution)):
            if bits[n] == "1":
                # Create a silicon rectangle for each "1" in the binary representation
                # fix the center traversing for the y part
                x = j * wg_resolution + wg_resolution/2 - rot_length/2
                y = i * wg_resolution + wg_resolution/2 - rot_width*1.5
                silicon = pf.Rectangle(size=(wg_resolution, wg_resolution), center=(x, y))
                hybrid_rotator_90.add("Si", silicon)
            n += 1

    hybrid_rotator_90.add_port(
        pf.Port(center=(-rot_length/2, 0), input_direction=0, spec=siepic_tech.ports["TE_1550_500"])
    )

    hybrid_rotator_90.add_port(
        pf.Port(center=(rot_length/2, 0), input_direction=180, spec=siepic_tech.ports["TE-TM_1550_500"])
    )

    field_monitor = td.FieldMonitor(
        center=(0, 0, 0.11), size=(td.inf, td.inf, 0), freqs=[freqs.mean()], name="field"
    )

    input_monitor = td.FieldMonitor(
        center=[-rot_length/2, 0.0, wg_height / 2],
        size=[0.0, rot_width * 3, wg_height * 3],
        freqs=freqs,
        name="input",
        fields=["Ex", "Ey", "Ez", "Hx", "Hy", "Hz"]
    )

    output_monitor = td.FieldMonitor(
        center=[rot_length/2, 0.0, wg_height / 2],
        size=[0.0, rot_width * 3, wg_height * 3],
        freqs=freqs,
        name="output",
        fields=["Ex", "Ey", "Ez", "Hx", "Hy", "Hz"]
    )

    # Include the Tidy3D simulation model
    hybrid_rotator_90.add_model(pf.Tidy3DModel(monitors=[field_monitor, input_monitor, output_monitor]), "Tidy3DModel")
    return hybrid_rotator_90


# Instantiate the component with custom dimensions

# you can change the num to see different digital metamaterial shapes
# max number = 2^max_pixels - 1
hybrid_rotator_90 = create_hybrid_rotator_90(num=202039409708820100, rot_length=5.0, rot_width=1.5, wg_resolution=0.1)
viewer(hybrid_rotator_90)

In [82]:
pf.tidy3d_plot(hybrid_rotator_90, plot_type="3d")